# Level 2 — Spatial Context: Mapping the Glioma Microenvironment

## CAJAL "Neuromics 2026" — Computational Mini-Project C10 (Level 2)

**Estimated time:** ~2.5 days

**Learning objectives**
- Load, QC, and explore Visium spatial transcriptomics data
- Get a first ("naive") spatial domain map directly from spot expression, before any deconvolution
- See the malignant cell-state axis directly in space, even before deconvolving spots
- Map your Level 1 reference onto tissue with **cell2location**
- Identify spatial niches (tissue domains) from the deconvolved cell-state map
- Quantify spatial organization: neighborhood enrichment, a proximity network, and spatial intermixing
- Compare your own results to the published figures (the paper is revealed in this notebook)

**Dataset:** Visium spatial transcriptomics from the **same two donors** as Level 1 — `AT10`
(primary, full feature set) and `AT14` (optional secondary section). Each spot covers
multiple cells (~1-10), so spot expression is a *mixture*, unlike Level 1's single nuclei.

> The paper is still not named yet. You'll recognize the cell-type and cell-state language
> from Level 1 — that continuity is the point. The reveal happens partway through this notebook.

---

## 0. Setup

In [ ]:
# Your code for: this step


## 1. Load and explore the spatial data

🔬 **TASK 1.1:** Load the AT10 Visium section and inspect the object — note how it differs from Level 1's AnnData (spatial coordinates, a tissue image, no per-nucleus QC metrics yet).

In [ ]:
# Your code for: Load the AT10 Visium section and inspect the object — note how it differs from Level 1's AnnData (spatial coordinates, a tissue image, no per-nucleus QC metrics yet)


In [ ]:
# Your code for: Load the AT10 Visium section and inspect the object — note how it differs from Level 1's AnnData (spatial coordinates, a tissue image, no per-nucleus QC metrics yet)


❓ **QUESTION:** Each Visium spot is ~55 µm in diameter. Given typical nucleus/cell sizes, roughly how many cells might a single spot cover? What does that imply about interpreting any single spot's gene expression?

## 2. Spatial quality control

Same idea as Level 1 — total counts, genes detected, %mito — but spots, not nuclei, and no doublet score (a "doublet" concept doesn't apply the same way to multi-cell spots).

🔬 **TASK 2.1:** Compute QC metrics and look at their distributions.

In [ ]:
# Your code for: Compute QC metrics and look at their distributions


In [ ]:
# Your code for: Compute QC metrics and look at their distributions


💡 **HINT:** Visium spots are much "deeper" than single nuclei (each spot pools several cells), so don't reuse Level 1's per-nucleus thresholds verbatim — look at *these* distributions. A light touch is usually right: drop near-empty spots (very low counts/genes), keep almost everything else.

🔬 **TASK 2.2:** Apply QC and a minimum-cells gene filter. Report spots/genes remaining.

In [ ]:
# Your code for: Apply QC and a minimum-cells gene filter. Report spots/genes remaining


⚠️ **CHECKPOINT:** This section should remove very few spots — on the order of **1-2% of spots** (this dataset is high quality; expect roughly **3,900-3,950 spots** remaining out of ~4,000, and somewhere around **24,000-25,000 genes** after the gene filter). If you lost a large fraction of spots, your thresholds are too strict for Visium-scale counts.

## 3. Normalization and a *naive* spatial domain map

Before any deconvolution, let's see what plain clustering of spot expression gives us — a "naive" map of spatial domains, mixing whatever cell types happen to co-occur in each spot.

🔬 **TASK 3.1:** Normalize, log-transform, select HVGs, and run PCA.

In [ ]:
# Your code for: Normalize, log-transform, select HVGs, and run PCA


🔬 **TASK 3.2:** Build the neighbor graph, UMAP, and cluster at a couple of resolutions. Pick one.

In [ ]:
# Your code for: Build the neighbor graph, UMAP, and cluster at a couple of resolutions. Pick one


💡 At resolution 0.5 you should see roughly **10-12** domains; at 1.0, roughly **16-20**. We carry resolution 1.0 forward as `leiden` — finer domains are easier to relate to the cell-state axis later, but feel free to use 0.5 instead.

🔬 **TASK 3.3:** Plot the naive domains both on the tissue and on UMAP, side by side.

In [ ]:
# Your code for: Plot the naive domains both on the tissue and on UMAP, side by side


❓ **QUESTION:** Do the naive domains form spatially coherent regions (contiguous patches of one color), or are they speckled/scattered across the tissue? What would each pattern imply about whether expression-based clustering alone is picking up real tissue architecture?

## 4. The malignant cell-state axis, in space (before deconvolution)

Level 1's marker-gene scoring works the same way here — `score_genes` doesn't care whether
an observation is a nucleus or a multi-cell spot. The catch: a spot's score is a *blend* of
whatever cell states happen to be in that spot, not a clean call.

🔬 **TASK 4.1:** Score every spot against the Level 1 malignant-state marker sets, using the shared `score_axis()` helper.

In [ ]:
# Your code for: Score every spot against the Level 1 malignant-state marker sets, using the shared `score_axis()` helper


🔬 **TASK 4.2:** Plot the paper's minimal 4-gene spatial zonation panel (`ZONATION_PANEL`: dev-like → gliosis → hypoxia) directly on tissue.

In [ ]:
# Your code for: Plot the paper's minimal 4-gene spatial zonation panel (`ZONATION_PANEL`: dev-like → gliosis → hypoxia) directly on tissue


❓ **QUESTION:** Do you see any spatial gradient across the four zonation genes — e.g. do `AQP4`-high and `HILPDA`-high regions occupy *different, non-overlapping* areas of the tissue? Compare this to the per-cluster axis scores above. This blended, spot-level picture is exactly the limitation **cell2location** is designed to address — keep this figure in mind for comparison once you've deconvolved.

---

## 5. Mapping single cells onto space with cell2location

So far every spot has been treated as one observation, even though it's really a mixture.
**cell2location** uses your Level 1 reference (cell types learned from single nuclei) to
estimate *how many cells of each type* are in every spot — turning "this spot's expression
looks bulk hypoxic" into "this spot is ~60% Hypoxic-state malignant cells + ~20% Macrophage + ...".

🔬 **TASK 5.1:** Load your saved, annotated Level 1 reference.

In [ ]:
# Your code for: Load your saved, annotated Level 1 reference


💡 **HINT — runtime.** cell2location is two models: a reference **signature** model (NB regression on your Level 1 single-cell counts) and a **spatial mapping** model (maps that signature onto spots). Both are slow on CPU. We benchmarked this on the actual data: training on all shared genes costs **~170s/epoch**; the standard cell2location gene filter (`cell2location.utils.filtering.filter_genes`, defaults) cuts that to **~15,900 genes** and **~72s/epoch** (reference) / **~3.9s/epoch** (spatial mapping, this Visium section's spot count). Paper-faithful epoch counts (ref 400 / mapping 6000) would take **~8 hours total on CPU** — only realistic on GPU. Set the mode below accordingly.

🔬 **TASK 5.2:** Set the compute mode, filter genes, and train the reference signature model.

💡 **HINT:** cell2location's reference model assumes a negative-binomial (GammaPoisson) likelihood over **raw integer counts** — but by this point your Level 1 reference's `.X` is log-normalized (from Level 1 Section 3). Point `setup_anndata` at the raw-counts layer explicitly (`layer="counts"`), or training will crash with a cryptic "value... not within the support of GammaPoisson" error the first time it tries to evaluate a likelihood on fractional log-values.

In [ ]:
# Your code for: Set the compute mode, filter genes, and train the reference signature model


In [ ]:
# Your code for: Set the compute mode, filter genes, and train the reference signature model


🔬 **TASK 5.3:** Train the spatial mapping model and export cell-type abundance per spot.

In [ ]:
# Your code for: Train the spatial mapping model and export cell-type abundance per spot


🔬 **TASK 5.4:** Plot a few cell-type abundance maps on tissue.

In [ ]:
# Your code for: Plot a few cell-type abundance maps on tissue


❓ **QUESTION:** Compare these deconvolved abundance maps to the blended Section 4 scores. Are the malignant-state spatial patterns sharper now? Does any region's *dominant* cell type surprise you given what the naive expression clustering (Section 3) suggested was there?

---

## 6. Spatial niches and intermixing

Individual cell-type abundances are noisy spot-by-spot. **NMF** on the abundance matrix finds
recurring *co-occurrence patterns* — niches — the way the original study does.

🔬 **TASK 6.1:** Run NMF on the cell2location abundance matrix with a few different factor counts.

In [ ]:
# Your code for: Run NMF on the cell2location abundance matrix with a few different factor counts


🔬 **TASK 6.2:** Plot niches on tissue, and look at which cell types load most strongly onto each niche factor.

In [ ]:
# Your code for: Plot niches on tissue, and look at which cell types load most strongly onto each niche factor


🔬 **TASK 6.3 — spatial intermixing.** For each spot, compute the Shannon entropy of its cell-type abundance distribution — a high-entropy spot has many cell types evenly mixed; a low-entropy spot is dominated by one.

In [ ]:
# Your code for: Plot niches on tissue, and look at which cell types load most strongly onto each niche factor


❓ **QUESTION:** Which niche has the lowest intermixing entropy (most "pure")? Which has the highest? Does low entropy correspond to a niche you'd expect to be compositionally homogeneous (e.g. a dense malignant core) versus a mixed immune/stromal region?

---

## 7. Spatial neighborhood analysis, two ways

🔬 **TASK 7.1 — squidpy.** Build the spatial neighbor graph and compute neighborhood enrichment between niches.

In [ ]:
# Your code for: Plot niches on tissue, and look at which cell types load most strongly onto each niche factor


🔬 **TASK 7.2 — the paper's own method.** Build the same kind of "which niches are near which" picture a different way: pairwise minimum spot-distance via a k-d tree, summarized at the 25th percentile (`gbmspace_utils.spatial_proximity_network` — this mirrors the paper's actual Fig. 2E/3C/6E/7E method, an alternative to squidpy's enrichment z-scores).

In [ ]:
# Your code for: Plot niches on tissue, and look at which cell types load most strongly onto each niche factor


❓ **QUESTION:** Do squidpy's neighborhood-enrichment z-scores and the proximity-distance heatmap agree on which niches are spatial neighbors? Where (if anywhere) do they disagree, and why might two reasonable methods for "spatial closeness" give different answers?

---

## 8. (Optional / stretch) Cell-cell communication

This section is optional — only attempt it if Sections 1-7 are solid and you have time left.

The published study runs **LIANA** (a consensus of several ligand-receptor methods) comparing
TME states co-localized with dev-like niches vs. those co-localized with gliosis/hypoxia
niches. We skip their cross-donor Tensor-cell2cell step (not meaningful with only 2 donors).

🔬 **TASK 8.1 (optional):** Using your own niches from Section 6, define two spot groups —
"dev-like-dominant" and "gliosis/hypoxia-dominant" — and run LIANA's consensus scoring on
the corresponding cells in your Level 1 reference, comparing ligand-receptor activity between
the two groups.

In [ ]:
# Your code for: Plot niches on tissue, and look at which cell types load most strongly onto each niche factor


## 9. Revealing the paper

📄 **de Jong, Memi, Gracia, Lazareva et al. "A spatiotemporal cancer cell trajectory
underlies glioblastoma heterogeneity." bioRxiv 2025.05.13.653495.** Companion website:
[gbmspace.org](https://www.gbmspace.org/). The data you have been working with (AT10 and
AT14, snRNA-seq + Visium) are two of the 12 IDH-wildtype glioblastoma tumours profiled in
this study (1,025,329 nuclei total across the cohort).

**Key findings, for comparison against your own results:**
- Malignant cells occupy a **continuous trajectory**, not discrete subtypes: from
  developmental-like states (OPC-like, NPC-neuronal-like, AC-progenitor-like) through a
  **gliosis** and **hypoxia** axis — what was historically called "mesenchymal-like" (MES1/2)
  in the Neftel et al. 2019 framework, but the authors show classical EMT regulators
  (`SNAI1/2`, `TWIST1/2`, `ZEB1/2`) are *not* specifically enriched there, arguing against an
  EMT interpretation.
- This trajectory maps onto **spatial zonation**: AC-progenitor-like cells dominate near the
  tumour core / infiltrating edge; gliosis and hypoxic states concentrate deep in the tumour,
  around and within necrotic regions — exactly the `AQP4` → `ABCC3` → `AKAP12` → `HILPDA`
  gradient you looked for in Section 4.
- Spatial **niches** were derived the same way you just did it: NMF on cell2location
  cell-state abundances (16 factors per tumour in the paper, clustered into ~14-16 recurrent
  niches across the cohort), cross-validated against pathologist-annotated IvyGAP regions
  (leading edge, infiltrating tumour, cellular tumour, necrosis, pseudopalisading cells,
  perinecrotic zone, microvascular proliferation).
- A major caveat the authors flag explicitly: **single-biopsy sampling can be misleading** —
  one of their tumours (AT10, the same donor in your data!) showed a different dominant
  malignant state in each of its 4 sampled sites, challenging the idea of one fixed
  "subtype" per tumour.

🔬 **TASK 9.1:** Now that you know the source, compare your own niche map and axis scores
against the paper'"'"'s actual cell2location/niche outputs for this exact section (these were
withheld from your input data — load them now from the answer-key file for comparison only).

In [ ]:
# Your code for: Now that you know the source, compare your own niche map and axis scores


❓ **QUESTION:** Where does your independent analysis agree with the published result, and where does it diverge? For any divergence, what's your best hypothesis — different gene filtering, different epoch budget (FAST vs FULL), different niche factor count, or something about how the reference was built in Level 1?

---

## 10. Write-up

🔬 **TASK 10.1:** Reproduce one specific published figure panel using your own pipeline, and write a short paragraph comparing your result to the original.

In [ ]:
# Your code for: Reproduce one specific published figure panel using your own pipeline, and write a short paragraph comparing your result to the original


## Summary

You have:
1. ✅ QC'd and explored real Visium spatial data
2. ✅ Built a naive (pre-deconvolution) spatial domain map
3. ✅ Seen the malignant cell-state axis directly in space
4. ✅ Mapped your Level 1 reference onto tissue with **cell2location**
5. ✅ Identified spatial niches via NMF, and quantified spatial intermixing
6. ✅ Compared two methods for spatial neighborhood/proximity analysis
7. ✅ Compared your independent results against the published findings

**Further reading, not built here:** the paper also describes **spaceTree** (joint cell-type
+ genetic-clone mapping) and **cell2fate** (RNA-velocity-based temporal ordering of malignant
states) — both require data/pipelines beyond this course's current scope (paired snATAC-seq
clone-calling, and spliced/unspliced counts, respectively).